# T17 박스 검출 YOLO nano 학습 (ADR-0004 2단계)

분할 검출기(`voss_vision.box_detect`, detector=seg)가 만든 **자동 라벨**로 YOLO nano 를 학습한다. 10/08 공용 PC 에서 같은 녹화로 분할과 비교한다.

1. 개인 PC 에서 만든 `t17_yolo.zip`(train + holdout + data.yaml)을 팀 Drive `datasets/t17_yolo_v1/` 에 올린다(구조: `docs/setup/drive.md`). 공유받은 팀원은 루트 폴더를 "내 드라이브에 바로가기 추가" 한다.
2. 런타임 → 유형 변경 → **GPU**. 위에서부터 차례로 실행.
3. 세션이 끊기면 1·2·3 셀을 다시 돌린 뒤 **이어 학습** 셀만 실행한다(체크포인트는 Drive 에 있다).

- holdout(C bag 2250 프레임~)은 학습에 쓰지 않는다. 사람 확인 200장 결과로 최종 비교한다.
- 가중치(`.pt`, `.onnx`)는 레포에 커밋하지 않는다(규칙 8). Drive `models/` 에만 둔다(이름에 버전 `_v1`).
- 지연(≤ 20 ms)은 Colab 이 아니라 공용 PC 에서 잰다.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
ROOT = "/content/drive/MyDrive/rokey_cobot2_VOSS"  # 팀 Drive 루트 (docs/setup/drive.md)
ZIP = f"{ROOT}/datasets/t17_yolo_v1/t17_yolo.zip"
RUNS = f"{ROOT}/colab/t17/runs"  # 체크포인트 (세션 끊김 대비)
MODELS = f"{ROOT}/models"
RUN_NAME = "t17_y11n"
MODEL = "yolo11n.pt"  # YOLO nano (ADR-0004)
OUT_NAME = "box_yolo11n_v1"  # models/ 에 둘 이름 (버전 접미사)

In [ ]:
!pip -q install ultralytics
!mkdir -p /content/data && cd /content/data && unzip -q -o "$ZIP" && ls t17_yolo
DATA = "/content/data/t17_yolo/data.yaml"

## 학습 (체크포인트 5 epoch 마다 Drive 저장)

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL)
model.train(
    data=DATA,
    imgsz=640,
    epochs=40,
    batch=16,
    seed=0,
    project=RUNS,
    name=RUN_NAME,
    exist_ok=True,
    save_period=5,
    fliplr=0.5,
    flipud=0.5,
    degrees=45,
    hsv_v=0.3,
)  # 박스 회전·밝기 변화 대비

## 이어 학습 (세션이 끊겼을 때만)

In [ ]:
from ultralytics import YOLO

YOLO(f"{RUNS}/{RUN_NAME}/weights/last.pt").train(resume=True)

## holdout 평가: 박스 프레임 검출률(IoU ≥ 0.5), 빈 프레임 오검출

In [ ]:
import glob
import os

from ultralytics import YOLO

best = YOLO(f"{RUNS}/{RUN_NAME}/weights/best.pt")
img_dir = "/content/data/t17_yolo/holdout/images"


def load(p):
    t = p.replace("/images/", "/labels/")[:-4] + ".txt"
    return [list(map(float, line.split()[1:])) for line in open(t) if line.strip()]


def iou(a, b):  # cx, cy, w, h (정규화)
    ax0, ay0, ax1, ay1 = a[0] - a[2] / 2, a[1] - a[3] / 2, a[0] + a[2] / 2, a[1] + a[3] / 2
    bx0, by0, bx1, by1 = b[0] - b[2] / 2, b[1] - b[3] / 2, b[0] + b[2] / 2, b[1] + b[3] / 2
    iw, ih = max(0, min(ax1, bx1) - max(ax0, bx0)), max(0, min(ay1, by1) - max(ay0, by0))
    inter = iw * ih
    return inter / (a[2] * a[3] + b[2] * b[3] - inter + 1e-9)


hit = n_box = fp = n_neg = 0
for p in sorted(glob.glob(f"{img_dir}/*.jpg")):
    gt = load(p)
    r = best.predict(p, imgsz=640, conf=0.5, verbose=False)[0]
    pred = r.boxes.xywhn.cpu().numpy().tolist()
    if gt:
        n_box += len(gt)
        hit += sum(any(iou(g, q) >= 0.5 for q in pred) for g in gt)
    else:
        n_neg += 1
        fp += bool(pred)
print(
    f"박스 검출률 {hit}/{n_box} = {100 * hit / max(n_box, 1):.2f}%  |  빈 프레임 오검출 {fp}/{n_neg}"
)
print(
    "주의: holdout 라벨도 자동 라벨이다. 최종 판정은 사람 확인 200장(holdout_check/check.csv) 기준."
)

## ONNX 내보내기 → Drive models/

In [ ]:
p = best.export(format="onnx", imgsz=640, opset=17, simplify=True)
os.makedirs(MODELS, exist_ok=True)
!cp "$p" "$MODELS/{OUT_NAME}.onnx" && ls -la "$MODELS"